In [2]:
import pandas as pd
import os

In [2]:
daily = pd.read_parquet("../data/all_addresses/all_addresses_anomalies.parquet")

In [3]:
# === CONFIG ===
output_folder = "../data/weka"
os.makedirs(output_folder, exist_ok=True)

# Columns to keep as features (numeric only)
key_features = [
    'normal_sent_cnt', 'normal_recv_cnt', 'normal_total_cnt',
    'eth_sent_sum', 'eth_recv_sum', 'eth_net_flow',
    'uniq_peers_cnt', 'sessions_cnt',
    'active_span_min', 'burst_max_tx_5m',
    'is_anomalous'
]

# ===== 1. SELECT ONLY NEEDED COLUMNS =====
weka_df = daily[key_features].copy()

weka_df['is_anomalous'] = weka_df['is_anomalous'].astype(int).astype(str)

# ===== 3. SAVE TO CSV =====
csv_path = os.path.join(output_folder, "anomaly_daily_weka.csv")
weka_df.to_csv(csv_path, index=False)
print(f"Saved WEKA CSV: {csv_path}")


Saved WEKA CSV: ../data/weka\anomaly_daily_weka.csv
Saved WEKA ARFF: ../data/weka\anomaly_daily_weka.arff


In [2]:
import pandas as pd

df = pd.read_parquet("../data/all_addresses/all_addresses_anomalies.parquet")

keep_cols = [
    'normal_sent_cnt', 'normal_recv_cnt', 'normal_total_cnt',
    'eth_sent_sum', 'eth_recv_sum', 'eth_net_flow',
    'uniq_peers_cnt', 'sessions_cnt',
    'active_span_min', 'burst_max_tx_5m',
    'is_anomalous'
]

df = df[keep_cols]

# -------------------------------------------
# 3. Balanced sampling
#    - Keep ALL anomalies
#    - Sample 200,000 normal rows
# -------------------------------------------
normal = df[df.is_anomalous == 0].sample(n=200_000, random_state=42)
anomalous = df[df.is_anomalous == 1]   # keep all anomalies

reduced_df = pd.concat([normal, anomalous], ignore_index=True)

# -------------------------------------------
# 5. Shuffle rows (important!)
# -------------------------------------------
reduced_df = reduced_df.sample(frac=1, random_state=42).reset_index(drop=True)

# -------------------------------------------
# 6. Print class counts
# -------------------------------------------
print("Class distribution:")
print(reduced_df['is_anomalous'].value_counts())

# -------------------------------------------
# 7. Export to CSV for WEKA
# -------------------------------------------
reduced_df.to_csv("weka_balanced.csv", index=False)

print("\nSaved balanced dataset as: weka_balanced.csv")


Class distribution:
is_anomalous
1    331472
0    200000
Name: count, dtype: int64

Saved balanced dataset as: weka_balanced.csv
